In [1]:
!pip3 install catboost

In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


from catboost import CatBoostRegressor

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [6]:
df = pd.read_csv("dsat.csv", dtype={352: "str", 353: "str"})

df = df[
    (df["DESIDR1_SPECTYPE"] != "b'STAR  '") &
    (df["ls_gaia_star_sn4"] == False) &
    (df["DESIDR1_SPECTYPE"].notna())
]

if "Unnamed: 0" in df.columns:
    df = df.drop(columns=["Unnamed: 0"])

In [8]:
import yaml

with open("used.yaml", "r") as f:
    config = yaml.safe_load(f)

features = config["features"]
print("Число признаков:", len(features))


print("Число используемых признаков:", len(features))

X = df[features]
y = df["DESIDR1_SPECTYPE"]

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=pd.qcut(y, q=5, labels=False)
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full,
    y_train_full,
    test_size=0.25,
    random_state=RANDOM_STATE
)

print("Train:", X_train.shape, "Val:", X_val.shape, "Test:", X_test.shape)

Число признаков: 232
Число используемых признаков: 232


TypeError: unsupported operand type(s) for -: 'str' and 'str'

In [ ]:
df = df.select_dtypes(include=["float64"])
print("Только float64:", df.shape)

df = df.replace([np.inf, -np.inf], np.nan)

df = df.fillna(df.mean())
print("После заполнения NaN средним:", df.shape)

In [ ]:
from itertools import product

param_grid = {
    "depth": [4, 6, 8],
    "learning_rate": [0.01, 0.03],
    "l2_leaf_reg": [1, 3, 5],
    "iterations": [1000],
}

cb_search_results = []

for depth, lr, l2, iters in product(
    param_grid["depth"],
    param_grid["learning_rate"],
    param_grid["l2_leaf_reg"],
    param_grid["iterations"],
):
    params = {
        "depth": depth,
        "learning_rate": lr,
        "l2_leaf_reg": l2,
        "iterations": iters,
        "loss_function": "RMSE",
        "random_state": RANDOM_STATE,
        "early_stopping_rounds": 100,
        "verbose": False,
    }

    model = CatBoostRegressor(**params)
    model.fit(X_train, y_train, eval_set=(X_val, y_val), use_best_model=True)

    y_val_pred = model.predict(X_val)
    metrics = evaluate_regression(
        y_val,
        y_val_pred,
        prefix=f"depth={depth}, lr={lr}, l2={l2}, iters={iters}"
    )

    cb_search_results.append({
        "depth": depth,
        "learning_rate": lr,
        "l2_leaf_reg": l2,
        "iterations": iters,
        "rmse_val": metrics["rmse"],
        "mae_val": metrics["mae"],
        "r2_val": metrics["r2"],
    })

cb_search_df = pd.DataFrame(cb_search_results).sort_values("rmse_val")
print("\nТоп-5 комбинаций CatBoost по RMSE на валидации:")
display(cb_search_df.head())

In [ ]:
best_row = cb_search_df.iloc[0]
best_cb_params = {
    "depth": int(best_row["depth"]),
    "learning_rate": float(best_row["learning_rate"]),
    "l2_leaf_reg": float(best_row["l2_leaf_reg"]),
    "iterations": int(best_row["iterations"]),
    "loss_function": "RMSE",
    "random_state": RANDOM_STATE,
    "early_stopping_rounds": 150,
    "verbose": 200,
}

print("Лучшие параметры CatBoost:", best_cb_params)

cb_best = CatBoostRegressor(**best_cb_params)
cb_best.fit(
    X_train_full,
    y_train_full,
    eval_set=(X_test, y_test),
    use_best_model=True,
)

y_test_pred_cb = cb_best.predict(X_test)
metrics_cb_test = evaluate_regression(
    y_test,
    y_test_pred_cb,
    prefix="\nCatBoost на тесте:"
)

importances = cb_best.get_feature_importance()
fi = pd.DataFrame({
    "feature": X_train_full.columns,
    "importance": importances,
}).sort_values("importance", ascending=False)

display(fi.head(20))

fi.to_csv("fi.txt", index=False, sep="\t")

In [ ]:
TOP_K = 30

top_features = fi.head(TOP_K)["feature"].tolist()
print(f"Топ-{TOP_K} признаков по важности:", top_features)

X_train_full_top = X_train_full[top_features]
X_train_top = X_train[top_features]
X_val_top = X_val[top_features]
X_test_top = X_test[top_features]

print("Train_full (top):", X_train_full_top.shape)
print("Train (top):", X_train_top.shape)
print("Val (top):", X_val_top.shape)
print("Test (top):", X_test_top.shape)

In [ ]:
k_values = list(range(1, 11, 1))
llr_results = []

for k in k_values:
    model = LocalLinearRegression(k=k)
    model.fit(X_train, y_train)
    y_val_pred = model.predict(X_val)
    metrics = evaluate_regression(y_val, y_val_pred, prefix=f"\nk={k}")
    
    llr_results.append({
        "k": k,
        "rmse_val": metrics["rmse"],
        "mae_val": metrics["mae"],
        "r2_val": metrics["r2"],
    })

llr_df = pd.DataFrame(llr_results).sort_values("rmse_val")
print("\nТоп-10 k для LLR (все признаки):")
display(llr_df.head(10))

best_k = int(llr_df.iloc[0]["k"])
print("\nЛучшее k для LLR (full features):", best_k)

llr_final = LocalLinearRegression(k=best_k)
llr_final.fit(X_train_full, y_train_full)
y_test_pred_llr = llr_final.predict(X_test)
metrics_llr_test = evaluate_regression(y_test, y_test_pred_llr,
                                       prefix="\nLLR (full features) на тесте:")

llr_final_top = LocalLinearRegression(k=best_k)
llr_final_top.fit(X_train_full_top, y_train_full)
y_test_pred_llr_top = llr_final_top.predict(X_test_top)
metrics_llr_test_top = evaluate_regression(
    y_test, y_test_pred_llr_top,
    prefix="\nLLR (top features) на тесте:"
)

In [ ]:
from sklearn.neural_network import MLPRegressor

hidden_configs = [
    (16, 8),
    (32, 16),
    (64, 32),
    (128, 64),
    (256, 128),
]

mlp_results = []

for hidden in hidden_configs:
    print(f"\nПробуем hidden_layer_sizes={hidden}")
    
    mlp = MLPRegressor(
        hidden_layer_sizes=hidden,
        activation="relu",
        solver="adam",
        alpha=1e-4,
        learning_rate="adaptive",
        learning_rate_init=1e-3,
        max_iter=10000,
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=100,
        random_state=RANDOM_STATE,
    )

    mlp.fit(X_train, y_train)
    y_val_pred = mlp.predict(X_val)

    metrics = evaluate_regression(y_val, y_val_pred,
                                  prefix=f"Val, hidden={hidden}")
    
    mlp_results.append({
        "hidden": hidden,
        "rmse_val": metrics["rmse"],
        "mae_val": metrics["mae"],
        "r2_val": metrics["r2"],
    })

mlp_df = pd.DataFrame(mlp_results).sort_values("rmse_val")
print("\nРезультаты перебора MLP (валидация):")
display(mlp_df)

best_hidden = mlp_df.iloc[0]["hidden"]
print("\nЛучший hidden_layer_sizes для MLP:", best_hidden)

mlp_final = MLPRegressor(
    hidden_layer_sizes=best_hidden,
    activation="relu",
    solver="adam",
    alpha=1e-4,
    learning_rate="adaptive",
    learning_rate_init=1e-3,
    max_iter=10000,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=100,
    random_state=RANDOM_STATE,
)

mlp_final.fit(X_train_full, y_train_full)
y_test_pred_mlp = mlp_final.predict(X_test)
metrics_mlp_test = evaluate_regression(
    y_test, y_test_pred_mlp,
    prefix="\nMLP (full features) на тесте:"
)

mlp_final_top = MLPRegressor(
    hidden_layer_sizes=best_hidden,
    activation="relu",
    solver="adam",
    alpha=1e-4,
    learning_rate="adaptive",
    learning_rate_init=1e-3,
    max_iter=10000,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=100,
    random_state=RANDOM_STATE,
)

mlp_final_top.fit(X_train_full_top, y_train_full)
y_test_pred_mlp_top = mlp_final_top.predict(X_test_top)
metrics_mlp_test_top = evaluate_regression(
    y_test, y_test_pred_mlp_top,
    prefix="\nMLP (top features) на тесте:"
)